# Q4 — 총정리: 데이터부터 결과까지
웨이퍼 결함 분류 4일 MVP의 전 과정을 한 노트북에 정리합니다.
데이터 시각화 → Q1/Q3 모델 비교 → Grad-CAM → 결론 순. 위에서부터 셀을 실행하세요.
전제: `Q1`, `Q3` 노트북을 먼저 돌려 체크포인트(`models/checkpoints/`)가 있어야 합니다.

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p = Path.cwd()
for cand in [_p, _p.parent, _p.parent.parent]:
    if (cand/"utils").is_dir():
        sys.path.insert(0, str(cand)); PROJ=cand; break
else:
    PROJ=_p
from utils.korean_font import set_korean_font
set_korean_font()

import numpy as np, matplotlib.pyplot as plt, itertools
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

CLASSES=["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM=len(CLASSES)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
PROC=PROJ/"data"/"processed"; CKPT=PROJ/"models"/"checkpoints"; FIG=PROJ/"results"/"figures"
FIG.mkdir(parents=True, exist_ok=True)
print("device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- 데이터 로드 ---
def load_split(n):
    d=np.load(PROC/f"wafer_{n}.npz", allow_pickle=True); return d["X"], d["y"]
Xtr,ytr=load_split("train"); Xte,yte=load_split("test")
counts=np.bincount(ytr, minlength=NUM)
print("train:", Xtr.shape, "| test:", Xte.shape)
print("값 0/1/2 = 배경 / 정상다이 / 불량다이")

## 1. 데이터 살펴보기
WM-811K 웨이퍼맵 데이터셋. 각 이미지는 128×128 격자이고 칸마다 0(배경)·1(정상 다이)·2(불량 다이) 값을 가집니다. 라벨은 9종(결함 8 + 정상 none).

In [ ]:
# --- 1-1. 클래스 분포 (불균형 확인) ---
fig, ax = plt.subplots(figsize=(9,4))
order=np.argsort(counts)
ax.barh([CLASSES[i] for i in order], counts[order], color="steelblue")
for k,i in enumerate(order):
    ax.text(counts[i], k, f"  {counts[i]:,} ({counts[i]/counts.sum()*100:.1f}%)", va="center", fontsize=9)
ax.set_title("클래스 분포 — 'none'(정상)이 85%, 심한 불균형")
plt.tight_layout(); plt.savefig(FIG/"q4_dist.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- 1-2. 클래스별 대표 웨이퍼맵 ---
fig, axes = plt.subplots(2,5, figsize=(14,6))
for k,ax in enumerate(axes.ravel()):
    if k>=NUM: ax.axis("off"); continue
    idx=np.where(ytr==k)[0][0]
    ax.imshow(Xtr[idx], cmap="viridis", vmin=0, vmax=2)
    ax.set_title(CLASSES[k], fontsize=10); ax.axis("off")
plt.suptitle("클래스별 대표 웨이퍼맵 — 결함마다 패턴이 다름", fontsize=13)
plt.tight_layout(); plt.savefig(FIG/"q4_samples.png", dpi=110, bbox_inches="tight"); plt.show()

## 2. 모델 두 개 불러와 평가
`Q1`(베이스라인)과 `Q3`(개선판) 체크포인트를 각각 불러와 동일한 test set으로 평가합니다. Q1은 inverse-freq 가중치(공격적), Q3는 Focal Loss + 균형 가중치(오탐 억제).

In [ ]:
# --- 2-1. 두 모델 평가 함수 ---
def build_net():
    n=models.resnet18(weights=None); n.fc=nn.Linear(n.fc.in_features, NUM); return n.to(DEVICE).eval()

def load_net(path):
    net=build_net(); net.load_state_dict(torch.load(path, map_location=DEVICE)["model"]); return net

@torch.no_grad()
def predict_all(net, X):
    out=[]
    for i in range(0,len(X),512):
        b=np.stack([(X[j].astype(np.float32)/2.0-0.5)/0.5 for j in range(i,min(i+512,len(X)))])
        b=np.expand_dims(b,1).repeat(3,1)
        out.append(net(torch.from_numpy(b).to(DEVICE)).argmax(1).cpu().numpy())
    return np.concatenate(out)

paths={"Q1":CKPT/"baseline_resnet18.pt", "Q3":CKPT/"q3_focal_resnet18.pt"}
preds={}; nets={}
for name,p in paths.items():
    if p.exists():
        nets[name]=load_net(p); preds[name]=predict_all(nets[name], Xte)
        acc=accuracy_score(yte,preds[name]); f1=f1_score(yte,preds[name],average="macro")
        print(f"{name}: accuracy={acc:.3f} | macro-F1={f1:.3f}")
    else:
        print(f"{name}: 체크포인트 없음 -> {p.name} (해당 노트북 먼저 실행)")

In [ ]:
# --- 2-2. 클래스별 precision/recall 비교 ---
fig, axes = plt.subplots(1,2, figsize=(15,5))
x=np.arange(NUM); w=0.35
for ax, metric, fn in zip(axes, ["precision","recall"], [precision_score, recall_score]):
    for off,name,color in [(-w/2,"Q1","#B4B2A9"),(w/2,"Q3","#185FA5")]:
        if name in preds:
            vals=fn(yte,preds[name],average=None,zero_division=0)
            ax.bar(x+off, vals, w, label=name, color=color)
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_ylim(0,1.05); ax.set_title(f"클래스별 {metric}"); ax.legend()
plt.suptitle("Q1(회색) vs Q3(파랑) — precision은 Q3가 크게 개선, recall은 소폭 트레이드오프", fontsize=12)
plt.tight_layout(); plt.savefig(FIG/"q4_pr_compare.png", dpi=110, bbox_inches="tight"); plt.show()

In [ ]:
# --- 2-3. 혼동행렬 나란히 비교 ---
avail=[n for n in ["Q1","Q3"] if n in preds]
fig, axes = plt.subplots(1, len(avail), figsize=(7*len(avail),6.5))
if len(avail)==1: axes=[axes]
for ax,name in zip(axes, avail):
    cm=confusion_matrix(yte,preds[name]); cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
    im=ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM)); ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticks(range(NUM)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel("예측"); ax.set_ylabel("실제")
    ax.set_title(f"{name}  (macro-F1={f1_score(yte,preds[name],average='macro'):.3f})")
    for i,j in itertools.product(range(NUM),range(NUM)):
        ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",
                color="white" if cmn[i,j]>0.5 else "black", fontsize=7)
plt.tight_layout(); plt.savefig(FIG/"q4_cm_compare.png", dpi=110, bbox_inches="tight"); plt.show()

## 3. 모델은 어디를 보고 판단하나 (Grad-CAM)
개선판(Q3)이 각 결함 클래스를 맞힐 때 웨이퍼의 어느 영역에 주목했는지 히트맵으로 봅니다. 빨간 영역이 판단 근거.

In [ ]:
# --- 3-1. Grad-CAM (Q3 모델, 결함 클래스별 1장씩) ---
best = nets.get("Q3", nets.get("Q1"))
class GradCAM:
    def __init__(s, m, layer):
        s.m=m; s.a=None; s.g=None
        layer.register_forward_hook(lambda mod,i,o: setattr(s,"a",o.detach()))
        layer.register_full_backward_hook(lambda mod,gi,go: setattr(s,"g",go[0].detach()))
    def __call__(s, x):
        s.m.zero_grad(); out=s.m(x); c=out.argmax(1).item(); out[0,c].backward()
        w=s.g.mean(dim=(2,3),keepdim=True)
        cam=F.relu((w*s.a).sum(1)).squeeze().cpu().numpy()
        cam=(cam-cam.min())/(cam.max()-cam.min()+1e-8)
        return cam, c, out.softmax(1)[0,c].item()
cam_engine=GradCAM(best, best.layer4)

def to_tensor(u8):
    img=(u8.astype(np.float32)/2.0-0.5)/0.5
    return torch.from_numpy(np.expand_dims(img,0).repeat(3,0)).unsqueeze(0)

defect=[c for c in range(NUM) if c!=8]
fig, axes=plt.subplots(2,len(defect), figsize=(2.1*len(defect),5))
for col,cls in enumerate(defect):
    idx=np.where(yte==cls)[0][0]
    cam,pred,prob=cam_engine(to_tensor(Xte[idx]).to(DEVICE))
    base=Xte[idx]/2.0
    axes[0,col].imshow(base, cmap="gray"); axes[0,col].axis("off")
    ok="O" if pred==cls else "X"
    axes[0,col].set_title(f"{CLASSES[cls]}\n예측 {ok} p={prob:.2f}", fontsize=8)
    axes[1,col].imshow(base, cmap="gray")
    axes[1,col].imshow(cam, cmap="jet", alpha=0.5, extent=(0,128,128,0), interpolation="bilinear")
    axes[1,col].axis("off")
plt.suptitle("Grad-CAM — 위:원본 / 아래:관심영역 (모델이 보는 곳)", fontsize=13)
plt.tight_layout(); plt.savefig(FIG/"q4_gradcam.png", dpi=110, bbox_inches="tight"); plt.show()

## 4. 결론 & 다음 단계


In [ ]:
# --- 4-1. 최종 요약표 출력 ---
print("="*52)
print(" 웨이퍼 결함 분류 — 4일 MVP 결과 요약")
print("="*52)
for name in [n for n in ['Q1','Q3'] if n in preds]:
    acc=accuracy_score(yte,preds[name]); f1=f1_score(yte,preds[name],average='macro')
    print(f"  {name:3s}  accuracy={acc:.3f}  macro-F1={f1:.3f}")
print("-"*52)
if "Q1" in preds and "Q3" in preds:
    d=f1_score(yte,preds['Q3'],average='macro')-f1_score(yte,preds['Q1'],average='macro')
    print(f"  개선 폭: macro-F1 {d:+.3f}")
    p1=precision_score(yte,preds['Q1'],average=None,zero_division=0)
    p3=precision_score(yte,preds['Q3'],average=None,zero_division=0)
    worst=int(np.argmax(p3-p1))
    print(f"  최대 precision 개선: {CLASSES[worst]}  {p1[worst]:.2f} -> {p3[worst]:.2f}")
print("="*52)

### 정리

이 프로젝트는 전처리된 웨이퍼맵 12만 장으로 결함 9종을 분류하는 모델을 4일 만에 만든 결과입니다.

- **Q1** 베이스라인(ResNet18)으로 돌아가는 모델과 첫 성능(macro-F1 0.811)을 확보했고,
- **Q2**에서 Grad-CAM으로 모델이 실제 결함 영역을 본다는 걸 확인했으며,
- **Q3**에서 Focal Loss + 균형 가중치로 오탐을 줄여 macro-F1 0.899까지 끌어올렸습니다.

`none`(정상)이 85%라 정확도(0.98)는 참고용이고, 실제 실력은 macro-F1과 클래스별 precision/recall로 봐야 합니다.

### 다음 디벨롭 후보
1. recall이 약간 내려간 Loc·Scratch·Edge-Loc → 오버샘플링 또는 gamma/beta 미세조정으로 균형점 찾기
2. Streamlit 추론 데모로 결과물화
3. 결함 원인 멀티태스크 헤드(Phase 2) — 결함 종류뿐 아니라 공정 원인까지 예측
4. EfficientNet 등 더 강한 백본 / 고해상도 실험